# 2.2 怎麼同時看前面幾個字？

# 第 2 章：讓模型看更長的上下文

前置：第1章的索引、loss、梯度。這章只增加固定上下文，沒有 attention。向量是可以學習的數字列表，不是先驗語意標籤。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：上下文像把幾張卡片一起攤開**

拼接順序保留位置；窗口固定長度。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/context.svg")))

**先想一想：**交換前兩個字會改變拼接結果嗎？

固定看前三個字，就把三個向量依順序接成一個長向量。順序由拼接位置保留，代價是 context 長度固定。

**把直覺寫成數學：**$[B,C,D]\to[B,C\cdot D]$；reshape 改擺法，不增加數值。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
ids = torch.tensor([[1, 2, 3], [3, 2, 1]])
embedding = nn.Embedding(4, 2)
context = embedding(ids).reshape(2, -1)
print(context.shape, context)
assert context.shape == (2, 6)

**如何讀結果：**B 是兩筆例子，C 是每筆可看字數，D 是特徵數，不要把軸名和數值混淆。

**只改一件事：**只交換第一筆的前兩個 ID。
